# LAB | Prompt Engineering Lab

## Part 1: Starting Simple - Create Initial Prompts

### Step 1: Environment Set-Up, Helper Functions

In [1]:
"""
This hands-on exercise will teach you the empirical science of prompt engineering by building production-ready prompts through testing, iteration, and refinement.
Author: Carlos Martinez Boto
"""
import os
from openai import OpenAI
from dotenv import load_dotenv
import re
import time
load_dotenv()

# import json
# from typing import List
# from pydantic import BaseModel

# Initialize client
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [2]:
MODEL = "gpt-4o-mini"

def call_openai(prompt, system_message="You are a helpful assistant.", temperature=0.7, max_tokens=None, seed=None):
    """Helper function to call OpenAI API with specified parameters"""
    try:
        params = {
            "model": MODEL,
            "messages": [
                {"role": "system", "content": system_message},
                {"role": "user", "content": prompt}
            ],
            "temperature": temperature
        }
        
        if max_tokens:
            params["max_tokens"] = max_tokens
        if seed is not None:
            params["seed"] = seed
            
        response = client.chat.completions.create(**params)
        return response.choices[0].message.content
    
    except Exception as e:
        return f"Error: {str(e)}"

def display_response(title, response, params=None):
    """Display a response with formatting"""
    print(f"\n{'='*60}")
    print(f"📝 {title}")
    if params:
        print(f"Parameters: {params}")
    print(f"{'='*60}")
    print(response)
    print(f"{'='*60}\n")

def count_tokens_approx(text):
    """Approximate token count (roughly 4 characters per token)"""
    return len(text) // 4

def print_dict_lists(data):
    for key, values in data.items():
        print(f"\n========= {key} =========")
        for item in values:
            print("\n---------")
            print(item)

def analyze_product_description(text: str) -> str:
    # Length of the string
    length = len(text)

    # Count price occurrences (e.g. $29.99)
    price_pattern = r"\$\d+(?:\.\d{2})?"
    price_count = len(re.findall(price_pattern, text))

    # Check if description title is simple
    description_simple = "Description:**" in text

    # Extract feature names between "- **" and "**:"
    feature_pattern = r"- \*\*(.*?)\*\*"
    features = re.findall(feature_pattern, text)

    features_count = len(features)

    # Normalize features to lowercase for counting
    features_lower = [f.lower() for f in features]

    # Count keyword occurrences in feature names
    wireless_count = sum("wireless" in f for f in features_lower)
    ergonomic_count = sum("ergonomic" in f for f in features_lower)
    tracking_count = sum("tracking" in f for f in features_lower)
    battery_count = sum("battery" in f for f in features_lower)
    stylish_count = sum("stylish" in f for f in features_lower)

    # Special handling for "plug" AND "play"
    plug_play_count = sum(("plug" in f and "play" in f) for f in features_lower)

    return (
        f"Length: {length}, price count: {price_count}, "
        f"description simple: {description_simple}, "
        f"features count: {features_count}, "
        f"wireless count: {wireless_count}, "
        f"ergonomic count: {ergonomic_count}, "
        f"tracking count: {tracking_count}, "
        f"battery count: {battery_count}, "
        f"plug/play count: {plug_play_count}, "
        f"stylish count: {stylish_count}"
    )

print("✅ Helper functions loaded!")

✅ Helper functions loaded!


### Step 2: Create Initial Prompts

Task 1: Sentiment Analysis (classification of customer messages):

In [3]:
# Initial simple prompt for sentiment analysis
sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""

# Test it once
result = call_openai(sentiment_prompt_v1)
print("Sentiment Analysis Result:")
print(result)

Sentiment Analysis Result:
The customer message can be classified as **Positive Feedback**.


Task 2: Product Description Generation

In [4]:
# Initial simple prompt for product description
product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""
 
# Test it once
result = call_openai(product_prompt_v1)
print("Product Description Result:")
print(result)
print(analyze_product_description(result))

Product Description Result:
**Product Description: Wireless Freedom Mouse - $29.99**

Experience the ultimate in convenience and comfort with our Wireless Freedom Mouse, your perfect companion for work or play. Priced at just $29.99, this sleek and stylish mouse combines cutting-edge technology with ergonomic design to elevate your computing experience.

**Key Features:**

- **Wireless Convenience:** Say goodbye to tangled cords! Our advanced 2.4GHz wireless technology ensures a stable connection with a range of up to 33 feet, allowing you to move freely without limitations.

- **Ergonomic Design:** Crafted with your comfort in mind, the Wireless Freedom Mouse features a contoured shape that fits perfectly in your hand, reducing strain during long hours of use. Its textured grips provide added control and comfort.

- **High Precision Tracking:** Equipped with a high-precision optical sensor, this mouse delivers smooth and accurate tracking on various surfaces, making it ideal for every

Task 3: Data Extraction

In [5]:
# Initial simple prompt for data extraction
extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""
 
# Test it once
result = call_openai(extraction_prompt_v1)
print("Data Extraction Result:")
print(result)

Data Extraction Result:
Here’s the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged


## Part 2: Diagnosing Failures - Systematic Testing


### Steps 3/4/5: Run Prompts 5/10/15 Times and Create Failure Analysis

Task 1: Sentiment Analysis

In [ ]:
task_1 = {}
for n in range(5, 16, 5):
    task_1[f'n_{n}'] = []
    for i in range(n):
        result = call_openai(sentiment_prompt_v1)
        task_1[f'n_{n}'].append(result)

In [10]:
print("Sentiment Analysis Result:")
print_dict_lists(task_1)

Sentiment Analysis Result:
========= n_5 =========
The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
The customer message can be classified as **Positive Feedback**.
The customer message can be classified as "Positive Feedback" or "Positive Review."
This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
========= n_10 =========
The customer message can be classified as **Positive Feedback** or **Satisfaction**.
This customer message can be classified as **positive feedback** or **satisfaction**.
This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
The customer message can be classified as **Positive Feedback** or **Positive Review**.
The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
The customer message can be classified as positive feedb

Task 2: Product Description Generation

In [11]:
task_2 = {}
for n in range(5, 16, 5):
    task_2[f'n_{n}'] = []
    for i in range(n):
        result = call_openai(product_prompt_v1)
        task_2[f'n_{n}'].append(result)

In [38]:
print("Product Description Result:")
for k in task_2.values():
    print("===================")
    for p in k:
        print(analyze_product_description(p))

Product Description Result:
Length: 1914, price count: 2, description simple: False, features count: 7, wireless count: 1, ergonomic count: 1, tracking count: 1, battery count: 1, plug/play count: 1, stylish count: 1
Length: 2018, price count: 2, description simple: True, features count: 7, wireless count: 1, ergonomic count: 1, tracking count: 1, battery count: 1, plug/play count: 1, stylish count: 1
Length: 1983, price count: 2, description simple: False, features count: 6, wireless count: 1, ergonomic count: 1, tracking count: 1, battery count: 1, plug/play count: 1, stylish count: 1
Length: 1928, price count: 2, description simple: False, features count: 6, wireless count: 1, ergonomic count: 1, tracking count: 1, battery count: 1, plug/play count: 1, stylish count: 0
Length: 1672, price count: 2, description simple: False, features count: 6, wireless count: 0, ergonomic count: 1, tracking count: 1, battery count: 1, plug/play count: 1, stylish count: 1
Length: 2067, price count: 2

Task 3: Data Extraction

In [13]:
task_3 = {}
for n in range(5, 16, 5):
    task_3[f'n_{n}'] = []
    for i in range(n):
        result = call_openai(extraction_prompt_v1)
        task_3[f'n_{n}'].append(result)

In [16]:
print("Data Extraction Result:")
print_dict_lists(task_3)

Data Extraction Result:
========= n_5 =========
---------
Here is the extracted information from the customer feedback:

- **Order Number**: 12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged
---------
Here is the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged
---------
Here is the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged
---------
Here is the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged
---------
Here is the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damage